In [11]:
import os

In [19]:
os.chdir('../')

In [20]:
%pwd


'd:\\Machine_Learning_Project\\Chicken-Disease-Classification-'

In [26]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [14]:
from CNNClassifier.constants import *
from CNNClassifier.utils.common import read_yaml, create_directories

In [23]:
class ConfigManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):

            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])

    def get_data_ingestion_cofig(self) -> DataIngestionConfig:
          config = self.config.data_ingestion

          create_directories([config.root_dir])

          data_ingestion_config = DataIngestionConfig(
                root_dir = config.root_dir,
                source_URL = config.source_URL,
                local_data_file = config.local_data_file,
                unzip_dir = config.unzip_dir
          )

          return data_ingestion_config

In [16]:
import os
import urllib.request as request
import zipfile
from CNNClassifier import logger
from CNNClassifier.utils.common import get_size

In [24]:
class DataIngestion:
    def __init__(self , config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download with following info: \n {headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            


In [27]:
try:
    config = ConfigManager()
    data_ingestion_config = config.get_data_ingestion_cofig()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-09-22 18:29:29,898: INFO: common]: yaml file: config\config.yaml loaded successfully]
[2026-09-22 18:29:29,903: INFO: common]: yaml file: params.yaml loaded successfully]
[2026-09-22 18:29:29,907: INFO: common]: Directory created at: artifacts]
[2026-09-22 18:29:29,911: INFO: common]: Directory created at: artifacts/data_ingestion]
[2026-09-22 18:29:32,639: INFO: 1411132918]: artifacts/data_ingestion/data.zip download with following info: 
 Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 8A4A:CBBCB:E7E11:1A98D6:6AB27BB3
x-github-edge-region: centralindia
Accept-Ranges: bytes
Date: Tue, 22 Sep 2026 12:59:32 GMT
Via: 1.1 varnish
X-S